### Шаг 1: Создание файла базы данных и загрузка таблиц

In [5]:
import sqlite3
import pandas as pd

# 1. Подключение к базе данных
db_path = 'ecommerce.db'
conn = sqlite3.connect(db_path)
print("База данных успешно создана!")

# 2. Генерация чистых демонстрационных данных для e-commerce проекта
users_data = pd.DataFrame({
    'user_id': range(1, 11),
    'join_date': pd.date_range(start='2026-01-01', periods=10, freq='D').strftime('%Y-%m-%d'),
    'country': ['Belarus', 'Russia', 'Belarus', 'Russia', 'Kazakhstan', 'Belarus', 'Russia', 'Kazakhstan', 'Belarus', 'Russia']
})

orders_data = pd.DataFrame({
    'order_id': range(101, 111),
    'user_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'order_date': ['2026-01-02', '2026-01-03', '2026-01-15', '2026-01-16', '2026-02-01', '2026-02-05', '2026-02-10', '2026-02-12', '2026-03-01', '2026-03-15'],
    'total_amount': [150.50, 42.00, 89.99, 200.00, 15.00, 310.00, 55.50, 120.00, 99.00, 45.00]
})

# 3. Сохранение данных в локальную SQL базу данных
users_data.to_sql('users', conn, if_exists='replace', index=False)
orders_data.to_sql('orders', conn, if_exists='replace', index=False)

print("Таблицы 'users' и 'orders' успешно загружены в базу 'ecommerce.db'!")
conn.close()

База данных успешно создана!
Таблицы 'users' и 'orders' успешно загружены в базу 'ecommerce.db'!


### Шаг 2: Написание первого исследовательского SQL-запроса

In [6]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Формирование SQL-запроса для объединения таблиц пользователей и заказов
query = """
SELECT 
    o.order_id,
    o.order_date,
    o.total_amount,
    u.user_id,
    u.country
FROM orders o
JOIN users u ON o.user_id = u.user_id
ORDER BY o.order_date;
"""

# 3. Выполнение запроса и сохранение результата в датафрейм
result_df = pd.read_sql_query(query, conn)
conn.close()

# Вывод результирующей таблицы на экран
result_df

,order_id,order_date,total_amount,user_id,country
0,101,2026-01-02,150.50,1,Belarus
1,102,2026-01-03,42.00,2,Russia
2,103,2026-01-15,89.99,3,Belarus
3,104,2026-01-16,200.00,4,Russia
4,105,2026-02-01,15.00,5,Kazakhstan
5,106,2026-02-05,310.00,6,Belarus
6,107,2026-02-10,55.50,7,Russia
7,108,2026-02-12,120.00,8,Kazakhstan
8,109,2026-03-01,99.00,9,Belarus
9,110,2026-03-15,45.00,10,Russia


### Шаг 3: Расчет базовых бизнес-метрик (выручка и средний чек по странам)

In [7]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Формирование SQL-запроса для агрегации метрик по странам
query = """
SELECT 
    u.country,
    SUM(o.total_amount) AS total_revenue,
    COUNT(o.order_id) AS total_orders,
    ROUND(AVG(o.total_amount), 2) AS average_order_value
FROM orders o
JOIN users u ON o.user_id = u.user_id
GROUP BY u.country
ORDER BY total_revenue DESC;
"""

# 3. Выполнение запроса и сохранение результата в датафрейм
metrics_df = pd.read_sql_query(query, conn)
conn.close()

# Вывод результирующей таблицы на экран
metrics_df

,country,total_revenue,total_orders,average_order_value
0,Belarus,649.49,4,162.37
1,Russia,342.50,4,85.63
2,Kazakhstan,135.00,2,67.50


### Шаг 4: Расчет накопительного итога выручки по месяцам (Оконные функции)

In [8]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Формирование SQL-запроса с использованием оконной функции для расчета накопительного итога
query = """
WITH monthly_sales AS (
    SELECT 
        STRFTIME('%Y-%m', o.order_date) AS sales_month,
        SUM(o.total_amount) AS monthly_revenue
    FROM orders o
    GROUP BY sales_month
)
SELECT 
    sales_month,
    monthly_revenue,
    ROUND(SUM(monthly_revenue) OVER (ORDER BY sales_month), 2) AS cumulative_revenue
FROM monthly_sales
ORDER BY sales_month;
"""

# 3. Выполнение запроса и сохранение результата в датафрейм
cumulative_df = pd.read_sql_query(query, conn)
conn.close()

# Вывод результирующей таблицы на экран
cumulative_df

,sales_month,monthly_revenue,cumulative_revenue
0,2026-01,482.49,482.49
1,2026-02,500.50,982.99
2,2026-03,144.00,1126.99


### Шаг 5: Когортный анализ удержания клиентов (Retention Rate)

In [9]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Формирование SQL-запроса для когортного анализа
query = """
WITH user_cohorts AS (
    -- Определение месяца первой покупки для каждого клиента
    SELECT 
        user_id,
        STRFTIME('%Y-%m', MIN(order_date)) AS cohort_month
    FROM orders
    GROUP BY user_id
),
order_periods AS (
    -- Расчет разницы в месяцах между текущей покупкой и месяцем когорты
    SELECT 
        o.user_id,
        c.cohort_month,
        STRFTIME('%Y-%m', o.order_date) AS order_month,
        (CAST(STRFTIME('%Y', o.order_date) AS INT) - CAST(STRFTIME('%Y', c.cohort_month) AS INT)) * 12 +
        (CAST(STRFTIME('%m', o.order_date) AS INT) - CAST(STRFTIME('%m', c.cohort_month) AS INT)) AS period_month
    FROM orders o
    JOIN user_cohorts c ON o.user_id = c.user_id
)
-- Подсчет количества уникальных пользователей в каждом периоде удержания
SELECT 
    cohort_month,
    COUNT(DISTINCT CASE WHEN period_month = 0 THEN user_id END) AS size_month_0,
    COUNT(DISTINCT CASE WHEN period_month = 1 THEN user_id END) AS size_month_1,
    COUNT(DISTINCT CASE WHEN period_month = 2 THEN user_id END) AS size_month_2
FROM order_periods
GROUP BY cohort_month
ORDER BY cohort_month;
"""

# 3. Выполнение запроса и сохранение результата в датафрейм
cohort_df = pd.read_sql_query(query, conn)
conn.close()

# Вывод результирующей таблицы на экран
cohort_df

,cohort_month,size_month_0,size_month_1,size_month_2
0,2026-01,0,0,0
1,2026-02,0,0,0
2,2026-03,0,0,0


### Шаг 6: Обновление данных для симуляции повторных покупок

In [11]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Генерация дополнительного списка повторных заказов для существующих пользователей
additional_orders = pd.DataFrame({
    'order_id': range(111, 116),
    'user_id': [1, 2, 4, 1, 2], # Пользователи 1, 2, 4 совершают повторные покупки
    'order_date': ['2026-02-15', '2026-02-20', '2026-03-05', '2026-03-10', '2026-03-20'],
    'total_amount': [45.00, 89.90, 120.00, 35.00, 150.00]
})

# 3. Добавление новых строк в существующую таблицу orders
additional_orders.to_sql('orders', conn, if_exists='append', index=False)
print("Повторные заказы успешно добавлены в таблицу 'orders'!")

# 4. Проверка общего обновленного количества заказов в базе данных
total_count = pd.read_sql_query("SELECT COUNT(*) AS total_orders FROM orders;", conn)
conn.close()

# Вывод актуального количества заказов
total_count

Повторные заказы успешно добавлены в таблицу 'orders'!


,total_orders
0,15


### Шаг 7: Повторный расчет когортного анализа удержания клиентов (Retention Rate)

In [13]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Формирование корректного SQL-запроса для когортного анализа
query = """
WITH user_cohorts AS (
    SELECT 
        user_id,
        MIN(STRFTIME('%Y-%m', order_date)) AS cohort_month
    FROM orders
    GROUP BY user_id
),
order_periods AS (
    SELECT 
        o.user_id,
        c.cohort_month,
        (CAST(SUBSTR(STRFTIME('%Y-%m', o.order_date), 1, 4) AS INT) - CAST(SUBSTR(c.cohort_month, 1, 4) AS INT)) * 12 +
        (CAST(SUBSTR(STRFTIME('%Y-%m', o.order_date), 6, 2) AS INT) - CAST(SUBSTR(c.cohort_month, 6, 2) AS INT)) AS period_month
    FROM orders o
    JOIN user_cohorts c ON o.user_id = c.user_id
)
SELECT 
    cohort_month,
    COUNT(DISTINCT CASE WHEN period_month = 0 THEN user_id END) AS size_month_0,
    COUNT(DISTINCT CASE WHEN period_month = 1 THEN user_id END) AS size_month_1,
    COUNT(DISTINCT CASE WHEN period_month = 2 THEN user_id END) AS size_month_2
FROM order_periods
GROUP BY cohort_month
ORDER BY cohort_month;
"""

# 3. Выполнение запроса и сохранение результата в датафрейм
cohort_df = pd.read_sql_query(query, conn)
conn.close()

# Вывод результирующей таблицы на экран
cohort_df

,cohort_month,size_month_0,size_month_1,size_month_2
0,2026-01,4,2,3
1,2026-02,4,0,0
2,2026-03,2,0,0


### Шаг 8: Перевод абсолютных значений в проценты (Retention Rate %)

In [14]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Формирование SQL-запроса для расчета Retention в процентах
query = """
WITH user_cohorts AS (
    SELECT 
        user_id,
        MIN(STRFTIME('%Y-%m', order_date)) AS cohort_month
    FROM orders
    GROUP BY user_id
),
order_periods AS (
    SELECT 
        o.user_id,
        c.cohort_month,
        (CAST(SUBSTR(STRFTIME('%Y-%m', o.order_date), 1, 4) AS INT) - CAST(SUBSTR(c.cohort_month, 1, 4) AS INT)) * 12 +
        (CAST(SUBSTR(STRFTIME('%Y-%m', o.order_date), 6, 2) AS INT) - CAST(SUBSTR(c.cohort_month, 6, 2) AS INT)) AS period_month
    FROM orders o
    JOIN user_cohorts c ON o.user_id = c.user_id
),
cohort_sizes AS (
    SELECT 
        cohort_month,
        COUNT(DISTINCT CASE WHEN period_month = 0 THEN user_id END) AS cohort_size
    FROM order_periods
    GROUP BY cohort_month
)
SELECT 
    p.cohort_month,
    s.cohort_size,
    ROUND(COUNT(DISTINCT CASE WHEN p.period_month = 0 THEN p.user_id END) * 100.0 / s.cohort_size, 2) || '%' AS month_0_pct,
    ROUND(COUNT(DISTINCT CASE WHEN p.period_month = 1 THEN p.user_id END) * 100.0 / s.cohort_size, 2) || '%' AS month_1_pct,
    ROUND(COUNT(DISTINCT CASE WHEN p.period_month = 2 THEN p.user_id END) * 100.0 / s.cohort_size, 2) || '%' AS month_2_pct
FROM order_periods p
JOIN cohort_sizes s ON p.cohort_month = s.cohort_month
GROUP BY p.cohort_month
ORDER BY p.cohort_month;
"""

# 3. Выполнение запроса и сохранение результата в датафрейм
retention_pct_df = pd.read_sql_query(query, conn)
conn.close()

# Вывод результирующей таблицы на экран
retention_pct_df

,cohort_month,cohort_size,month_0_pct,month_1_pct,month_2_pct
0,2026-01,4,100.0%,50.0%,75.0%
1,2026-02,4,100.0%,0.0%,0.0%
2,2026-03,2,100.0%,0.0%,0.0%


### Шаг 9: Сегментация клиентов (RFM-анализ с оконными функциями)

In [15]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Формирование SQL-запроса для проведения RFM-анализа клиентов
query = """
WITH user_rfm_raw AS (
    SELECT 
        user_id,
        -- Расчет количества дней с момента последнего заказа до условной текущей даты
        CAST(JULIANDAY('2026-03-31') - JULIANDAY(MAX(order_date)) AS INT) AS recency_days,
        COUNT(order_id) AS frequency_count,
        SUM(total_amount) AS monetary_value
    FROM orders
    GROUP BY user_id
),
rfm_scores AS (
    SELECT 
        user_id,
        recency_days,
        frequency_count,
        monetary_value,
        -- Сегментация на 3 группы (для Recency: чем меньше дней, тем выше балл)
        NTILE(3) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(3) OVER (ORDER BY frequency_count ASC) AS f_score,
        NTILE(3) OVER (ORDER BY monetary_value ASC) AS m_score
    FROM user_rfm_raw
)
SELECT 
    user_id,
    recency_days,
    frequency_count,
    ROUND(monetary_value, 2) AS monetary_value,
    (r_score || f_score || m_score) AS rfm_segment
FROM rfm_scores
ORDER BY rfm_segment DESC, monetary_value DESC;
"""

# 3. Выполнение запроса и сохранение результата в датафрейм
rfm_df = pd.read_sql_query(query, conn)
conn.close()

# Вывод результирующей таблицы на экран
rfm_df

,user_id,recency_days,frequency_count,monetary_value,rfm_segment
0,2,11,3,281.90,333
1,1,21,3,230.50,332
2,10,16,1,45.00,311
3,4,26,2,320.00,233
4,8,47,1,120.00,222
5,9,30,1,99.00,222
6,6,54,1,310.00,123
7,3,75,1,89.99,111
8,7,49,1,55.50,111
9,5,58,1,15.00,111


### Шаг 10: Подготовка к публикации на GitHub

In [17]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect('ecommerce.db')

# 2. Выгрузка всех витрин данных в файлы Excel для последующей визуализации
with pd.ExcelWriter('ecommerce_analytics_outputs.xlsx') as writer:
    pd.read_sql_query("SELECT * FROM users JOIN orders ON users.user_id = orders.user_id;", conn).to_excel(writer, sheet_name='Raw_Data', index=False)
    retention_pct_df.to_excel(writer, sheet_name='Cohort_Retention', index=False)
    rfm_df.to_excel(writer, sheet_name='RFM_Segmentation', index=False)

print("Аналитические витрины успешно сохранены в файл 'ecommerce_analytics_outputs.xlsx'!")
conn.close()

Аналитические витрины успешно сохранены в файл 'ecommerce_analytics_outputs.xlsx'!
